# Transformer Decoder

## Code

In [1]:
import torch
import torch.nn as nn

import models.deep_learning.architectures as mynn
import models.deep_learning.components as comp

## Testing

In [2]:
# input parameters
N = 3
M = 4
batch_size = 2
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
dtype = torch.float32

# Decoder Layer parameters
d_model = 4
nhead = 2
dim_feedforward = 64
dropout = 0.2
layer_norm_eps = 1e-5
batch_first = True
norm_first = True
bias = True

tgt_mask = comp.create_causal_mask((N, N), device=device)
memory_mask = comp.create_random_mask((N, M), device=device)


## Transformer decoder parameters
num_layers = 5
#  It helps only when norm_first is True,
norm = None  # nn.LayerNorm(d_model).to(device=device,dtype=dtype)

In [3]:
torch.manual_seed(0)
x = torch.randn(batch_size, N, d_model, device=device, dtype=dtype)
memory = torch.randn(batch_size, M, d_model, device=device, dtype=dtype)

In [4]:
init_seed = 42  # avoide weights initialization randomness effects
train_seed = 24  # avoid dropout randomness effects

In [5]:
torch.manual_seed(init_seed)
tf_decl = mynn.TransformerDecoderLayer(
    d_model,
    nhead,
    dim_feedforward=dim_feedforward,
    dropout=dropout,
    activation_cls=nn.GELU,
    layer_norm_eps=layer_norm_eps,
    norm_first=norm_first,
    bias=bias,
    device=device,
    dtype=dtype,
)

torch.manual_seed(init_seed)
nn_tf_decl = nn.TransformerDecoderLayer(
    d_model,
    nhead,
    dim_feedforward=dim_feedforward,
    dropout=dropout,
    activation="gelu",
    layer_norm_eps=layer_norm_eps,
    batch_first=batch_first,
    norm_first=norm_first,
    bias=bias,
    device=device,
    dtype=dtype,
)

tf_dec = mynn.TransformerDecoder(tf_decl, num_layers, norm=norm)
nn_tf_dec = nn.TransformerDecoder(
    nn_tf_decl,
    num_layers,
    norm=norm,
)

tf_dec.load_weights_from_torch_decoder(nn_tf_dec)

### Evaluation

In [6]:
tf_dec.eval()
tf_dec(x, memory, tgt_mask=tgt_mask, memory_mask=memory_mask)

tensor([[[-1.2885, -2.0291,  3.3357, -2.4961],
         [-0.2115, -1.5903,  3.6717, -1.1551],
         [ 1.5087,  3.3349, -0.4980,  4.9272]],

        [[-0.7925,  0.4295, -1.1579, -0.8034],
         [ 1.1971,  0.5541, -1.3229, -0.0244],
         [ 2.7689,  2.2613,  2.4942,  2.1989]]], device='mps:0',
       grad_fn=<AddBackward0>)

In [7]:
nn_tf_dec.eval()
nn_tf_dec(x, memory, tgt_mask=tgt_mask, memory_mask=memory_mask)

tensor([[[-1.2885, -2.0291,  3.3357, -2.4961],
         [-0.2115, -1.5903,  3.6717, -1.1551],
         [ 1.5087,  3.3349, -0.4980,  4.9272]],

        [[-0.7925,  0.4295, -1.1579, -0.8034],
         [ 1.1971,  0.5541, -1.3229, -0.0244],
         [ 2.7689,  2.2613,  2.4942,  2.1989]]], device='mps:0',
       grad_fn=<AddBackward0>)

### Training

In [8]:
mse = torch.nn.MSELoss()

In [9]:
torch.manual_seed(train_seed)
nn_tf_dec.train()
out = nn_tf_dec(x, memory)
print(out)
loss = mse(out, x)
loss.backward()
optimizer = torch.optim.SGD(nn_tf_dec.parameters(), lr=1e-3)
optimizer.step()
nn_tf_dec(x, memory)


tensor([[[ 0.0831, -1.0748,  1.1178, -1.3246],
         [ 0.7610,  1.0347,  0.4498,  1.1008],
         [ 0.0937,  0.9695,  0.9439,  1.1212]],

        [[-0.5338,  2.8519, -1.1166,  2.3343],
         [ 0.2728,  1.0747, -0.3270,  1.5421],
         [ 2.0798,  1.5716,  0.8372, -0.2262]]], device='mps:0',
       grad_fn=<AddBackward0>)


tensor([[[-1.1066, -0.5696,  0.4400, -1.3393],
         [ 0.4884,  0.6554,  1.1871,  0.6520],
         [ 0.4244, -0.0606,  1.5420, -0.8693]],

        [[-0.4925,  0.7962,  0.1237,  2.1709],
         [ 0.2391,  0.8243, -1.2805, -0.0617],
         [ 2.5202,  1.9464,  1.7192,  1.1238]]], device='mps:0',
       grad_fn=<AddBackward0>)

In [10]:
torch.manual_seed(train_seed)
tf_dec.train()
out = tf_dec(x, memory)
print(out)
loss = mse(out, x)
loss.backward()
optimizer = torch.optim.SGD(tf_dec.parameters(), lr=1e-3)
optimizer.step()
tf_dec(x, memory)

tensor([[[ 0.0831, -1.0748,  1.1178, -1.3246],
         [ 0.7610,  1.0347,  0.4498,  1.1008],
         [ 0.0937,  0.9695,  0.9439,  1.1212]],

        [[-0.5338,  2.8519, -1.1166,  2.3343],
         [ 0.2728,  1.0747, -0.3270,  1.5421],
         [ 2.0798,  1.5716,  0.8372, -0.2262]]], device='mps:0',
       grad_fn=<AddBackward0>)


tensor([[[-1.1061, -0.5693,  0.4405, -1.3389],
         [ 0.4889,  0.6558,  1.1872,  0.6523],
         [ 0.4245, -0.0606,  1.5422, -0.8691]],

        [[-0.4922,  0.7960,  0.1241,  2.1714],
         [ 0.2393,  0.8244, -1.2798, -0.0612],
         [ 2.5210,  1.9469,  1.7193,  1.1245]]], device='mps:0',
       grad_fn=<AddBackward0>)